# 2주차 학습 코드

실제 제출 모델은 제공 라벨로 학습하였다. 직접 주석 라벨 대체는 기본값 False로 두었다.
원래 실험의 분할 함수와 학습 설정을 보관한 코드이다. 원본 계열 접미사 처리 누락은 최종 보고서 6절과 results/source_family_overlap.csv에 기록하였다.


In [ ]:
USE_MANUAL_ANNOTATIONS = False  # 이번 best.pt는 제공 라벨로 학습하였다.
# 0. 2주차 준비
%pip install -q ultralytics

import random
import re
from collections import Counter
from pathlib import Path
from zipfile import ZipFile

import torch
import yaml
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = Path("/content/drive/MyDrive/딥러닝")
ZIP_PATH = DRIVE_DIR / "사용한 데이터.zip"
MANUAL_ZIP = DRIVE_DIR / "step3_week2_manual.zip"
WORK_DIR = Path("/content/step3_week2")
RESULT_DIR = DRIVE_DIR / "step3_week2"
EXTRACT_DIR = WORK_DIR / "dataset"
CLASS_NAMES = {0: "crack", 1: "efflorescence", 2: "spall_detach"}
SEED = 42

# 1주차 ZIP을 같은 런타임에서 한 번만 압축 해제한다.
if not ZIP_PATH.is_file():
    raise FileNotFoundError(f"실습 ZIP을 확인하세요: {ZIP_PATH}")

zip_stat = ZIP_PATH.stat()
zip_version = f"{zip_stat.st_size}:{zip_stat.st_mtime_ns}"
marker = EXTRACT_DIR / ".source_zip"
if not marker.is_file():
    with ZipFile(ZIP_PATH) as archive:
        archive.extractall(EXTRACT_DIR)
    marker.write_text(zip_version)
elif marker.read_text() != zip_version:
    raise RuntimeError("실습 ZIP이 바뀌었습니다. 새 런타임에서 실행하세요.")

SRC_ROOT = next(
    (p.parent for p in EXTRACT_DIR.rglob("images") if p.is_dir() and (p.parent / "labels").is_dir()),
    None,
)
if SRC_ROOT is None:
    raise FileNotFoundError("압축을 푼 폴더에서 images와 labels를 찾지 못했습니다.")

SRC_IMG_DIR = SRC_ROOT / "images"
SRC_LBL_DIR = SRC_ROOT / "labels"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

def check_class_names(source):
    names = yaml.safe_load(source)["names"]
    names = dict(enumerate(names)) if isinstance(names, list) else {int(k): v for k, v in names.items()}
    if names != CLASS_NAMES:
        raise ValueError(f"클래스 ID 순서 확인 필요: {names}")

source_yaml = SRC_ROOT / "data.yaml"
if source_yaml.is_file():
    check_class_names(source_yaml.read_text(encoding="utf-8"))

def remove_export_suffix(stem):
    return re.sub(r"(?:_(?:jpg|jpeg|png))?\.rf\.[0-9a-f]+(?=\.|_|$)", "",
                  stem, flags=re.IGNORECASE)

def get_origin_id(stem):
    return re.split(r"_(aug|rot|flip|contrast|brightness|crop|noise|blur|scale|shift)",
                    remove_export_suffix(stem), maxsplit=1, flags=re.IGNORECASE)[0]

def is_augmented(stem):
    return remove_export_suffix(stem) != get_origin_id(stem)

manual_labels = {}
if USE_MANUAL_ANNOTATIONS and MANUAL_ZIP.is_file():
    # 직접 주석 라벨을 파일명으로 연결한다.
    image_stems = [p.stem for p in SRC_IMG_DIR.rglob("*.jpg")]
    exact_counts = Counter(image_stems)
    by_name = {}
    for stem in image_stems:
        by_name.setdefault(remove_export_suffix(stem), []).append(stem)

    with ZipFile(MANUAL_ZIP) as archive:
        for entry in archive.infolist():
            member = Path(entry.filename)
            if member.name in {"data.yaml", "dataset.yaml"}:
                check_class_names(archive.read(entry))
            if "labels" not in member.parts[:-1] or member.suffix.lower() != ".txt":
                continue
            stem = member.stem
            matches = [stem] if exact_counts[stem] == 1 else by_name.get(remove_export_suffix(stem), [])
            if len(matches) != 1 or matches[0] in manual_labels:
                raise ValueError(f"직접 주석 파일명과 원본 이미지 확인 필요: {member.name}")
            manual_labels[matches[0]] = archive.read(entry).decode("utf-8-sig")

    if not manual_labels:
        raise ValueError("직접 주석 ZIP의 labels 폴더에서 TXT를 찾지 못했습니다.")
    print("직접 주석 영상:", len(manual_labels), "장")
else:
    # 기존 train에서 원본 ID가 다른 주석 후보를 최대 15장 추출한다.
    candidates = {}
    for image in sorted((SRC_IMG_DIR / "train").rglob("*.jpg")):
        if not is_augmented(image.stem):
            candidates.setdefault(get_origin_id(image.stem), image)

    if candidates:
        picked = random.Random(SEED).sample(list(candidates.values()), min(15, len(candidates)))
        polygon_zip = WORK_DIR / "polygon_input.zip"
        with ZipFile(polygon_zip, "w") as archive:
            for image in picked:
                archive.write(image, image.name)
        print(f"주석 후보 {len(picked)}장:", polygon_zip)
    else:
        print("train 폴더에서 직접 주석할 원본 후보를 선택하세요.")

print(f"이미지 {len(list(SRC_IMG_DIR.rglob('*.jpg'))):,}장 | 라벨 {len(list(SRC_LBL_DIR.rglob('*.txt'))):,}개")
print("CUDA:", torch.cuda.is_available())

## 1. 포맷 확인 · 강의자료 PDF 11쪽

제출용으로 라벨 파일 경로와 첫 줄이 함께 보이는 출력을 남긴다. 첫 라벨이 비어 있으면 첫 번째 비어 있지 않은 라벨을 확인한다.

In [ ]:
import glob

lbls = sorted(glob.glob(str(SRC_LBL_DIR / "**" / "*.txt"), recursive=True))
print("라벨 파일 수:", len(lbls))

first_label = None
line = []
for p in lbls:
    with open(p, encoding="utf-8-sig") as f:
        line = f.readline().split()
    if line:
        first_label = p
        break

if first_label is None:
    raise ValueError("첫 줄을 확인할 수 있는 라벨 파일이 없습니다.")

print("라벨 파일:", Path(first_label).relative_to(SRC_ROOT))
print("첫 줄:", " ".join(line))
print("토큰 개수:", len(line))
print("앞부분:", line[:9])

if len(line) == 5:
    print("Detection 포맷입니다.")
elif len(line) >= 7 and (len(line) - 1) % 2 == 0:
    print("Segmentation 포맷입니다.")
else:
    print("라벨 형식을 다시 확인하세요.")

## 2. 데이터 무결성 점검 · 강의자료 PDF 13쪽

이미지와 라벨은 split을 포함한 상대 경로와 파일명으로 연결한다. 좌표 개수, 클래스 ID, 좌표 범위를 확인한다. 빈 라벨은 배경으로 의도된 것인지 확인한 뒤, 확인한 파일만 `CONFIRMED_BACKGROUND`에 입력한다. 출력된 이상 항목은 원본을 확인하여 수정한 다음 이 셀을 다시 실행한다. 제출 노트북에는 점검 결과와 수정 여부를 남긴다.

In [ ]:
images = sorted(SRC_IMG_DIR.rglob("*.jpg"))
image_keys = {p.relative_to(SRC_IMG_DIR).with_suffix("").as_posix(): p for p in images}
label_keys = {Path(p).relative_to(SRC_LBL_DIR).with_suffix("").as_posix(): Path(p)
              for p in lbls}

missing_labels = sorted(image_keys.keys() - label_keys.keys())
orphan_labels = sorted(label_keys.keys() - image_keys.keys())

# 출력과 같은 형식으로 입력한다. 예: {"train/frame001"}
CONFIRMED_BACKGROUND = set()
bad = set()
empty_labels = []
manual_instances = Counter()

def check_label(text):
    counts = Counter()
    for ln in text.splitlines():
        v = ln.split()
        if not v:
            continue
        try:
            cls = int(v[0])
            if cls not in CLASS_NAMES or len(v) < 7 or (len(v) - 1) % 2 != 0:
                return None
            if any(not 0 <= float(x) <= 1 for x in v[1:]):
                return None
        except ValueError:
            return None
        counts[cls] += 1
    return counts

for key, p in label_keys.items():
    text = manual_labels.get(p.stem)
    if text is None:
        text = p.read_text(encoding="utf-8-sig")
    counts = check_label(text)
    if counts is None:
        bad.add(key)
    elif not counts:
        empty_labels.append(key)
        if p.stem in manual_labels:
            bad.add(key)
    elif p.stem in manual_labels:
        manual_instances.update(counts)

unconfirmed_empty = sorted(set(empty_labels) - CONFIRMED_BACKGROUND)
print("이미지:", len(images), "| 라벨:", len(lbls))
print("라벨 없는 영상:", len(missing_labels), missing_labels[:10])
print("영상 없는 라벨:", len(orphan_labels), orphan_labels[:10])
print("빈 라벨:", len(empty_labels))
print("배경 확인이 필요한 빈 라벨:", len(unconfirmed_empty), unconfirmed_empty[:10])
print("포맷/클래스/좌표 이상 파일:", len(bad), sorted(bad)[:10])
print("직접 주석 영상:", len(manual_labels))
print("직접 주석 클래스별 인스턴스:",
      {name: manual_instances[i] for i, name in CLASS_NAMES.items()})

if missing_labels or orphan_labels or bad or unconfirmed_empty:
    raise ValueError("이상 항목의 원인을 확인하고 수정한 뒤 다시 실행하세요.")

## 3. 원본 단위 분할과 data.yaml · 강의자료 PDF 12·14·15쪽

강의자료의 원본 ID 그룹화 및 `80:10:10` 목표 비율을 사용한다. 파일명으로 확인한 증강은 학습셋에만 포함한다. 검증과 테스트로 선택된 원본 그룹의 증강본은 복사에서 제외한다. 원본 후보가 없는 그룹과 직접 주석한 그룹은 train에 배정하므로 실제 이미지 수 비율은 달라질 수 있다. 이전에 제공된 split과 별개로 이 코드에서 다시 분할한다. **3주차에도 이 분할을 사용하므로 `split_manifest.csv`를 저장한다.**

In [ ]:
import csv
import shutil

if not manual_labels:
    print("직접 주석 ZIP이 없어 제공 라벨로 분할합니다. 제출용 직접 주석 실적은 별도로 준비하세요.")
if missing_labels or orphan_labels or bad or unconfirmed_empty:
    raise ValueError("무결성 점검 이상 항목을 수정한 뒤 다시 실행하세요.")

SPLIT_RATIO = (0.8, 0.1, 0.1)
OUT_ROOT = WORK_DIR / "dataset_split"
random.seed(SEED)

# 제출에 표시할 핵심 코드 1: 원본 ID가 같은 영상끼리 묶는다.
groups = {}
for img_path in images:
    origin_id = get_origin_id(img_path.stem)
    groups.setdefault(origin_id, []).append(img_path)

if not groups:
    raise FileNotFoundError("실습 데이터에서 JPG를 찾지 못했습니다.")

origin_ids = sorted(groups)
random.shuffle(origin_ids)
eligible = [origin_id for origin_id in origin_ids
            if any(not is_augmented(img.stem) for img in groups[origin_id])
            and all(img.stem not in manual_labels for img in groups[origin_id])]

# 제출에 표시할 핵심 코드 2: 그룹 단위로 val/test를 배정한다.
n_val = int(len(origin_ids) * SPLIT_RATIO[1])
n_test = int(len(origin_ids) * SPLIT_RATIO[2])
if min(n_val, n_test) < 1 or len(eligible) < n_val + n_test:
    raise ValueError("검증·테스트에 배정할 원본 ID가 부족합니다.")

val_ids = set(eligible[:n_val])
test_ids = set(eligible[n_val:n_val + n_test])
train_ids = set(origin_ids) - val_ids - test_ids
split_ids = {"train": train_ids, "val": val_ids, "test": test_ids}

# 재실행 시 이전 split의 파일이 섞이지 않게 작업용 폴더만 다시 만든다.
if OUT_ROOT.exists():
    shutil.rmtree(OUT_ROOT)

manifest = []
excluded_aug = 0
image_counts = Counter()
for split, ids in split_ids.items():
    img_dir = OUT_ROOT / "images" / split
    lbl_dir = OUT_ROOT / "labels" / split
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    for origin_id in sorted(ids):
        for img_path in groups[origin_id]:
            augmented = is_augmented(img_path.stem)
            if split != "train" and augmented:
                excluded_aug += 1
                continue

            key = img_path.relative_to(SRC_IMG_DIR).with_suffix("").as_posix()
            dst_img = img_dir / img_path.relative_to(SRC_IMG_DIR)
            dst_lbl = lbl_dir / Path(key + ".txt")
            dst_img.parent.mkdir(parents=True, exist_ok=True)
            dst_lbl.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(img_path, dst_img)
            if img_path.stem in manual_labels:
                dst_lbl.write_text(manual_labels[img_path.stem], encoding="utf-8")
            else:
                shutil.copy2(label_keys[key], dst_lbl)

            image_counts[split] += 1
            manifest.append((split, origin_id, key, augmented,
                             dst_img.relative_to(OUT_ROOT).as_posix()))

written_ids = {split: {row[1] for row in manifest if row[0] == split} for split in split_ids}
overlaps = {
    "train/val": len(written_ids["train"] & written_ids["val"]),
    "train/test": len(written_ids["train"] & written_ids["test"]),
    "val/test": len(written_ids["val"] & written_ids["test"]),
}
eval_aug = sum(row[3] for row in manifest if row[0] != "train")
assert not any(overlaps.values()) and eval_aug == 0
print("split별 원본 ID 수:", {split: len(ids) for split, ids in written_ids.items()})
print("split별 이미지 수:", dict(image_counts))
print("split 간 원본 ID 중복:", overlaps)
print("val/test 내 증강 표시 이미지:", eval_aug)
print("val/test 그룹에서 제외한 증강본:", excluded_aug)

data_yaml = {
    "path": str(OUT_ROOT.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": CLASS_NAMES,
}
YAML_PATH = OUT_ROOT / "data.yaml"
YAML_PATH.write_text(yaml.safe_dump(data_yaml, allow_unicode=True, sort_keys=False),
                     encoding="utf-8")
shutil.copy2(YAML_PATH, RESULT_DIR / "data.yaml")
with open(RESULT_DIR / "split_manifest.csv", "w", newline="", encoding="utf-8-sig") as file:
    writer = csv.writer(file)
    writer.writerow(["split", "origin_id", "source_key", "augmented", "output_image"])
    writer.writerows(manifest)

print(YAML_PATH.read_text(encoding="utf-8"))

## 4. YOLOv8s-Seg 학습 실행 · 강의자료 PDF 16쪽

제출용은 `epochs=60`으로 실행한다. 동작만 시험할 때는 `SMOKE_TEST=True`로 3 epoch를 별도 저장할 수 있다. 첫 epoch에 `box_loss`, `seg_loss`, `cls_loss`, `dfl_loss`가 나오는지 확인한다. CUDA 메모리가 부족하면 `batch=2`로 수정한다. Drive의 해당 학습 폴더에 `weights/best.pt`가 저장된다.

In [ ]:
from ultralytics import YOLO

if not torch.cuda.is_available():
    raise RuntimeError("Colab 런타임 유형을 GPU로 변경하고 준비 셀부터 실행하세요.")

SMOKE_TEST = False
model = YOLO("yolov8s-seg.pt")
model.train(
    data=str(YAML_PATH),
    epochs=3 if SMOKE_TEST else 60,
    imgsz=960,
    batch=4,
    optimizer="AdamW",
    lr0=0.00015,
    mosaic=0.0,
    mixup=0.0,
    patience=20,
    seed=42,
    device=0,
    project=str(RESULT_DIR / "runs"),
    name="step3_smoke" if SMOKE_TEST else "step3",
)

RUN_DIR = Path(model.trainer.save_dir)
BEST_PT = RUN_DIR / "weights" / "best.pt"
if not BEST_PT.is_file():
    raise FileNotFoundError(f"best.pt 저장 여부를 확인하세요: {BEST_PT}")
print("best.pt:", BEST_PT)
print("학습 기록:", RUN_DIR / "results.csv")

## 5. 학습 곡선 확인 · 강의자료 PDF 17쪽

학습 셀과 동일한 런타임에서 실행한다. 새 런타임에서 그림만 다시 만들 경우 `RUN_DIR`에 실제 학습 결과가 저장된 Drive 폴더 경로를 지정한다.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

df = pd.read_csv(RUN_DIR / "results.csv")
df.columns = [c.strip() for c in df.columns]

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# ① 학습 / 검증 Segmentation loss
for c in ["train/seg_loss", "val/seg_loss"]:
    if c in df.columns:
        ax[0].plot(df["epoch"], df[c], label=c)

ax[0].set_title("Segmentation Loss")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("Loss")
ax[0].legend()
plt.close(fig)  # 다음 셀에서 두 그래프를 함께 표시한다.

## 6. 학습 곡선 확인 코드(2) · 강의자료 PDF 18쪽

바로 위 셀에서 만든 그림에 Mask mAP를 추가하고 `curve.png`로 저장한다.

In [ ]:
# ② Mask mAP
for c in ["metrics/mAP50(M)", "metrics/mAP50-95(M)"]:
    if c in df.columns:
        ax[1].plot(df["epoch"], df[c], label=c)

ax[1].set_title("Mask mAP")
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("mAP")
ax[1].legend()

fig.tight_layout()
fig.savefig(RUN_DIR / "curve.png", dpi=150)
display(fig)

print("학습 곡선:", RUN_DIR / "curve.png")
print("제출할 가중치:", RUN_DIR / "weights" / "best.pt")

## 7. 이번 주 제출물 · 강의자료 PDF 22쪽

| 항목 | 넣을 내용 |
|---|---|
| 1. 주석 결과 | 직접 그린 영상 수와 `crack`, `efflorescence`, `spall_detach`별 인스턴스 수를 적는다. 한 영상에 결함이 3개 있으면 인스턴스는 3개로 센다. |
| 2. 포맷 확인 | 라벨 파일 첫 줄이 보이는 포맷 확인 셀의 출력 화면을 넣는다. |
| 3. 무결성 점검 | 이미지·라벨 매칭, 빈 라벨, 포맷·클래스·좌표 점검 결과와 발견한 이상 항목의 수정 여부를 적는다. |
| 4. 분할 코드 | **원본 ID 추출, 그룹화, 그룹 단위 split 배정 코드**에 표시한다. split별 원본 ID·이미지 수와 중복 점검 결과를 넣는다. |
| 5. 학습 곡선 | 최종 학습에서 만든 `curve.png` 또는 `results.png`를 넣고, segmentation loss와 mask mAP 변화가 실제로 어땠는지 짧게 적는다. |
| 6. 설정 근거 | 아래 문장을 참고해 `imgsz=960`, Mosaic·MixUp 비활성화 이유를 적는다. |

> 가는 균열의 세부 형태를 최대한 보존하기 위해 입력 크기를 960으로 설정하였다. Mosaic은 서로 다른 영상의 연결로 콘크리트 표면의 공간적 맥락이 달라질 수 있어 비활성화하였다. MixUp은 서로 다른 영상이 겹치면서 실제 결함 경계가 흐려질 수 있어 비활성화하였다.

다음 주 강의 전까지 **PDF 또는 실행 출력이 담긴 노트북 1개**, **최종 학습의 `best.pt`**, **실습 코드·결과 파일**을 낸다. 실습 파일은 `data.yaml`, `curve.png` 또는 `results.png`, `results.csv`, `split_manifest.csv`를 함께 정리하면 된다. 3주차 평가를 위해 직접 주석 ZIP과 `split_manifest.csv`도 Drive에 보관한다.